<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)
print(dataset["train"][0])

In [ ]:
count = 0

for text in dataset["train"]["text"]:
    if text and text.strip():
        count += 1

print("Non-empty docs:", count)

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "mdsalmanhossain/bangla-social-media-abuse-dataset"
)

print(path)

In [ ]:
import pandas as pd
import os

file = os.path.join(path, os.listdir(path)[0])

df = pd.read_excel(file)

print(df.columns.tolist())
print("Rows:", len(df))

In [ ]:
import random

TARGET = 62617
random.seed(42)

reservoir = []
seen = 0

for text in dataset["train"]["text"]:

    if not text or not text.strip():
        continue

    seen += 1

    if len(reservoir) < TARGET:
        reservoir.append(text.strip())
    else:
        j = random.randint(0, seen - 1)

        if j < TARGET:
            reservoir[j] = text.strip()

print("News:", len(reservoir))

In [ ]:
social_docs = (
    df["comment"]
    .dropna()
    .astype(str)
    .tolist()
)

social_62617 = []

while len(social_62617) < 62617:
    social_62617.extend(social_docs)

social_62617 = social_62617[:62617]

print("Social:", len(social_62617))

In [ ]:
combined = reservoir + social_62617

random.shuffle(combined)

print("Total:", len(combined))

In [ ]:
with open(
    "news_social_125k.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in combined:
        f.write(doc.strip() + "\n")

print("Saved")

In [ ]:
import os

print(
    "Documents:",
    sum(
        1
        for _ in open(
            "news_social_125k.txt",
            encoding="utf-8"
        )
    )
)

print(
    "Size MB:",
    round(
        os.path.getsize(
            "news_social_125k.txt"
        )/(1024**2),
        2
    )
)

In [ ]:
from collections import Counter

chars = 0
words = 0
vocab = Counter()

with open(
    "news_social_125k.txt",
    encoding="utf-8"
) as f:

    for line in f:

        chars += len(line)

        toks = line.split()

        words += len(toks)

        vocab.update(toks)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))
print("TTR        :", len(vocab)/words)
print(
    "Average Word Length:",
    chars/words
)

In [ ]:
from sklearn.model_selection import train_test_split

with open(
    "news_social_125k.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

train_docs, test_docs = train_test_split(
    docs,
    test_size=0.10,
    random_state=42
)

print("Train:", len(train_docs))
print("Test :", len(test_docs))

In [ ]:
with open(
    "train_news_social_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in train_docs:
        f.write(doc + "\n")

with open(
    "test_news_social_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in test_docs:
        f.write(doc + "\n")

print("Saved")

In [ ]:
import os

print(
    "Train MB:",
    round(
        os.path.getsize(
            "train_news_social_bn.txt"
        )/(1024**2),
        2
    )
)

print(
    "Test MB:",
    round(
        os.path.getsize(
            "test_news_social_bn.txt"
        )/(1024**2),
        2
    )
)

Transliteration

In [ ]:
!pip install aksharamukha
from aksharamukha import transliterate

In [ ]:
def transliterate_file(
    infile,
    outfile
):

    count = 0

    with open(
        infile,
        encoding="utf-8"
    ) as fin, open(
        outfile,
        "w",
        encoding="utf-8"
    ) as fout:

        for line in fin:

            line = line.strip()

            if not line:
                continue

            iso = transliterate.process(
                "Bengali",
                "ISO",
                line
            )

            fout.write(
                iso + "\n"
            )

            count += 1

            if count % 10000 == 0:

                print(
                    f"{count:,} docs"
                )

    print(
        "Completed:",
        count
    )

In [ ]:
transliterate_file(
    "train_news_social_bn.txt",
    "train_news_social_iso.txt"
)

In [ ]:
transliterate_file(
    "test_news_social_bn.txt",
    "test_news_social_iso.txt"
)

In [ ]:
import random

with open(
    "test_news_social_bn.txt",
    encoding="utf-8"
) as f:

    bn_lines = [
        x.strip()
        for x in f
        if x.strip()
    ]

with open(
    "test_news_social_iso.txt",
    encoding="utf-8"
) as f:

    iso_lines = [
        x.strip()
        for x in f
        if x.strip()
    ]

random.seed(42)

idxs = random.sample(
    range(len(bn_lines)),
    100
)

correct = 0

for idx in idxs:

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso_lines[idx]
    )

    if back == bn_lines[idx]:
        correct += 1

print(
    "Exact Match:",
    correct,
    "/ 100"
)

print(
    "Accuracy:",
    correct/100
)

In [ ]:
import random

random.seed(42)

shown = 0

for idx in idxs:

    original = bn_lines[idx]

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso_lines[idx]
    )

    if original != back:

        print("="*80)
        print("ORIGINAL:")
        print(original)

        print("\nBACK:")
        print(back)

        shown += 1

        if shown == 10:
            break

In [ ]:
import os
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0
    vocab = Counter()

    with open(path, encoding="utf-8") as f:

        for line in f:

            chars += len(line)

            toks = line.split()

            words += len(toks)

            vocab.update(toks)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

bn = corpus_stats("train_news_social_bn.txt")
iso = corpus_stats("train_news_social_iso.txt")

print("Bengali:", bn)
print("ISO:", iso)

print(
    "\nExpansion Factor:",
    iso["chars"] / bn["chars"]
)

print(
    "Vocabulary Ratio:",
    iso["vocab"] / bn["vocab"]
)

Tokenization

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = (
        Whitespace()
    )

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_bn_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

In [ ]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_bn_{vocab_size}.json",
        "test_news_social_bn.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_iso_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

In [ ]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [ ]:
for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_iso_{vocab_size}.json",
        "test_news_social_iso.txt"
    )

    print(r)

print("\nFinished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_bn_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

In [ ]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_news_social_bn.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_iso_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

In [ ]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [ ]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_news_social_iso.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    spm.SentencePieceTrainer.train(
        input="train_news_social_bn.txt",
        model_prefix=f"uni_bn_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0
    )

    print(
        "Saved:",
        vocab_size
    )

In [ ]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(
    model_path,
    test_file
):

    sp = spm.SentencePieceProcessor()
    sp.load(model_path)

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                pieces = sp.encode(
                    w,
                    out_type=str
                )

                words += 1
                tokens += len(pieces)
                chars += len(w)

                lengths.append(
                    len(pieces)
                )

                if len(pieces) > 1:
                    fragmented += 1

                if "<unk>" in pieces:
                    oov += 1

    return {
        "vocab": sp.get_piece_size(),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [ ]:
for vocab_size in VOCABS:

    r = evaluate_unigram(
        f"uni_bn_{vocab_size}.model",
        "test_news_social_bn.txt"
    )

    print(r)

print("\nFinished.")

In [ ]:
import sentencepiece as spm

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    spm.SentencePieceTrainer.train(
        input="train_news_social_iso.txt",
        model_prefix=f"uni_iso_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0
    )

    print(
        "Saved:",
        vocab_size
    )

In [ ]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(
    model_path,
    test_file
):

    sp = spm.SentencePieceProcessor()
    sp.load(model_path)

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                pieces = sp.encode(
                    w,
                    out_type=str
                )

                words += 1
                tokens += len(pieces)
                chars += len(w)

                lengths.append(
                    len(pieces)
                )

                if len(pieces) > 1:
                    fragmented += 1

                if "<unk>" in pieces:
                    oov += 1

    return {
        "vocab": sp.get_piece_size(),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [ ]:
for vocab_size in VOCABS:

    r = evaluate_unigram(
        f"uni_iso_{vocab_size}.model",
        "test_news_social_iso.txt"
    )

    print(r)

print("\nFinished.")

## News + Social Media Corpus

The News+Social corpus combines formal news articles with informal social media comments, providing a balanced representation of structured and user-generated Bengali text. The final corpus contained 125,234 documents, evenly distributed between the two domains. The corpus consisted of approximately 3.89 million words and 343,964 unique vocabulary items, with a type-token ratio (TTR) of 0.0883. Compared to the standalone social media corpus, the mixed corpus exhibited a lower TTR due to the higher frequency of recurring lexical items inherited from the news domain.

Transliteration from Bengali script to ISO 15919 increased corpus size by approximately 16.6%, producing an expansion factor of 1.166. The ISO representation reduced the observed vocabulary size by roughly 3.8%, resulting in a vocabulary ratio of 0.962. This behavior closely matches observations from the news and literature corpora, indicating that script transformation primarily affects character-level representation rather than lexical diversity.

A transliteration sanity check produced an exact-match accuracy of 52%. Manual inspection revealed that most mismatches were caused by orthographic normalization, punctuation substitutions, Unicode representation differences, and alternate spellings rather than semantic corruption. Therefore, the exact-match score underestimates the practical fidelity of the transliteration process.

Across all vocabulary sizes, BPE consistently achieved the best intrinsic performance among the evaluated tokenization methods. At a vocabulary size of 50,000, BPE produced the lowest fertility (1.283), lowest word fragmentation ratio (0.208), and highest characters-per-token value (4.145) in Bengali script. WordPiece showed slightly higher fertility and fragmentation, while Unigram exhibited the highest fragmentation and lowest compression efficiency.

The ISO representation demonstrated behavior similar to the Bengali script. Fertility and fragmentation metrics remained nearly unchanged across scripts, while CPT values increased substantially due to the longer character sequences introduced by ISO transliteration. This pattern suggests that script representation primarily influences token length and compression characteristics rather than segmentation behavior.

Overall, the News+Social corpus reinforced the trends observed in previous experiments. BPE remained the strongest tokenizer across all intrinsic metrics, WordPiece consistently ranked second, and Unigram produced the least compact segmentations. The results further indicate that the superiority of BPE is robust across both formal and informal Bengali text domains.